In [36]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from TTS.utils.manage import ModelManager
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.glow_tts_config import GlowTTSConfig
from TTS.tts.models.glow_tts import GlowTTS
from TTS.utils.audio import AudioProcessor
from trainer.callbacks import TrainerCallback
import json

# Enable MPS fallback for macOS
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
audio_model_config_path = "dataset/audio_model_config.json"

class Config:
    """Configuration optimized for M3 Pro"""
    data_path = "./dataset/chunks"
    output_path = "./tts_output"
    
    # Using GlowTTS - most stable for fine-tuning
    pretrained_model = "tts_models/en/ljspeech/glow-tts"
    
    # M3 Pro optimized settings
    batch_size = 16
    eval_batch_size = 8
    num_loader_workers = 4
    
    # Training settings
    lr = 1e-4
    epochs = 100
    steps_to_generate = 5000
    save_step = 2500
    log_step = 500
    save_n_checkpoints = 10
    restore_path = ""
    best_path = ""
    use_ddp = False
    use_accelerate = False
    grad_accum_steps = 1
    overfit_batch = False
    skip_train_epoch = False
    start_with_eval = False
    small_run = 0

def preprocess_data(config):
    """Load and validate dataset"""
    metadata = []
    
    for file in os.listdir(config.data_path):
        if file.endswith(".csv"):
            csv_path = os.path.join(config.data_path, file)
            try:
                df = pd.read_csv(csv_path, names=["file", "text"])
                
                for _, row in df.iterrows():
                    wav_path = os.path.join(config.data_path, row["file"])
                    
                    if (os.path.exists(wav_path) and 
                        wav_path.endswith(".wav") and 
                        len(str(row["text"]).strip()) > 0 and
                        os.path.getsize(wav_path) > 1000):
                        
                        metadata.append({
                            "text": str(row["text"]).strip(),
                            "audio_file": wav_path
                        })
                        
            except Exception as e:
                print(f"Error processing {csv_path}: {str(e)}")
    
    if len(metadata) == 0:
        raise ValueError("No valid audio/text pairs found!")
    
    return metadata

class SampleGenerationCallback(TrainerCallback):
    """Generate audio samples during training"""
    def __init__(self, config, ap):
        self.config = config
        self.ap = ap
        self.test_phrases = [
            "This is a test synthesis.",
            "The model is learning your voice.",
            "Fine tuning is in progress."
        ]
    
    def on_train_step_end(self, trainer, model, outputs):
        if trainer.global_step % self.config.steps_to_generate == 0:
            print(f"Generating samples at step {trainer.global_step}...")
            
            try:
                model.eval()
                with torch.no_grad():
                    for i, text in enumerate(self.test_phrases):
                        try:
                            output = model.inference(text)
                            wav = output["wav"].squeeze().cpu().numpy()
                            
                            out_path = os.path.join(
                                trainer.output_path,
                                f"step_{trainer.global_step}_sample_{i}.wav"
                            )
                            self.ap.save_wav(wav, out_path)
                            
                        except Exception:
                            continue
                            
                model.train()
                print("Sample generation completed")
                
            except Exception:
                pass

def main():
    config = Config()
    os.makedirs(config.output_path, exist_ok=True)
    
    print("Starting TTS fine-tuning...")
    print(f"Using MPS: {torch.backends.mps.is_available()}")
    
    # Download pretrained model
    print("Downloading pretrained model...")
    manager = ModelManager()
    model_path, _, _ = manager.download_model(config.pretrained_model)
    
    # Load ORIGINAL config from pretrained model
    model_config = GlowTTSConfig()
    model_config.load_json(audio_model_config_path)
    
    # ONLY modify training parameters - keep architecture/audio params from original
    model_config.batch_size = config.batch_size
    model_config.eval_batch_size = config.eval_batch_size
    model_config.num_loader_workers = config.num_loader_workers
    model_config.run_eval = True
    model_config.epochs = config.epochs
    model_config.lr = config.lr
    model_config.mixed_precision = True
    
    # Training control parameters
    model_config.save_step = config.save_step
    model_config.print_step = config.log_step
    model_config.save_n_checkpoints = config.save_n_checkpoints
    model_config.save_best = True
    
    # Prepare dataset
    print("Loading dataset...")
    metadata = preprocess_data(config)
    train_samples, eval_samples = train_test_split(metadata, test_size=0.1, random_state=42)
    print(f"Dataset loaded: {len(train_samples)} training, {len(eval_samples)} validation samples")
    
    # Initialize model from updated config
    print("Initializing model...")
    model = GlowTTS.init_from_config(model_config)
    
    # Load pretrained weights with strict=False to ignore mismatches
    print("Loading pretrained weights...")
    try:
        checkpoint = torch.load(model_path, map_location="cpu")
        state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
        model.load_state_dict(state_dict, strict=False)
        print("Pretrained weights partially loaded (strict=False)")
    except Exception as e:
        print(f"Warning: Could not load pretrained weights: {e}")
        print("Training from scratch...")
    
    # Initialize audio processor
    ap = AudioProcessor.init_from_config(model_config)
    
    # Setup trainer with CORRECTED TrainerArgs
    trainer_args = TrainerArgs(
        continue_path="",  # Empty for new training
        restore_path="",   # Not restoring from specific checkpoint
        best_path="",      # No best model yet
        use_ddp=False,
        use_accelerate=False,
        grad_accum_steps=1,
        overfit_batch=False,
        skip_train_epoch=False,
        start_with_eval=False,
        small_run=0,
    )
    
    trainer = Trainer(
        trainer_args,
        config=model_config,
        output_path=config.output_path,
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
        callbacks=[SampleGenerationCallback(config, ap)],
    )
    
    # Start training
    print(f"Starting training with {len(train_samples)} samples...")
    try:
        trainer.fit()
        print("Training completed successfully!")
    except KeyboardInterrupt:
        print("Training stopped")
    except Exception as e:
        print(f"Training failed: {str(e)}")
        raise

if __name__ == "__main__":
    main()

Starting TTS fine-tuning...
Using MPS: True
 > tts_models/en/ljspeech/glow-tts is already downloaded.
Loading dataset...
Dataset loaded: 1684 training, 188 validation samples
Initializing model...
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:0
 | > fft_size:1024
 | > power:1.1
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:50.0
 | > mel_fmax:7600.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
Loading pretrained weights...
Pretrained weights partiall

OSError: [Errno 45] Operation not supported: '/home/erogol'